# Does fine-tuning help with literary translation?

In the preceding notebook, we fine-tuned Llama 3.2 (3B) on English–Dutch sentence pairs from Agatha Christie’s *The ABC Murders*. Here we translate the same 100 sentences from another detective novel, Arthur Conan Doyle’s *The Sign of the Four*, in two ways.

- **Zero-shot** uses the original Llama model with no examples in the prompt.
- **Fine-tuned** uses the same Llama model with the LoRA adapter saved after training. The adapter is the small set of additional weights learned during fine-tuning.

Both runs use the same English sentences, prompt and generation settings. The Dutch reference translations are only used for scoring. This comparison uses Llama for both runs so we can examine the effect of fine-tuning. The introductory notebook used EuroLLM.

## 1. Set up Colab and find the saved adapter

Run the fine-tuning notebook first. It saves the adapter and tokenizer to **My Drive → lora_model**.

Open this notebook in a fresh Colab session. Under **Runtime → Change runtime type**, select a **T4 GPU**, then choose **Run all**. Allow Colab to access Drive when prompted. The first run downloads the model and the text.

In [1]:
%pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.12.0 bitsandbytes==0.49.2 sacrebleu==2.6.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
adapter_dir = Path("/content/drive/MyDrive/lora_model")
sentence_count = 100  # Try 5 for a quick run.

if not (adapter_dir / "adapter_config.json").is_file():
    raise FileNotFoundError(
        f"Run the fine-tuning notebook first to save the adapter to {adapter_dir}."
    )

Mounted at /content/drive


## 2. Load the English sentences and their Dutch references

We use the first 100 one-to-one aligned sentence pairs from Chapter I, after the chapter heading. An aligned pair links an English sentence to its published Dutch translation, *De Agra-schat*. We skip bracketed notes and use the same pairs each time. Automatic alignment can still introduce mismatches, and the Dutch edition uses older spellings such as *zijne*. These can affect the scores. [OPUS Books, aligned by Andras Farkas](https://opus.nlpl.eu/datasets/Books). [Corpus citation: Jörg Tiedemann (2012)](https://aclanthology.org/L12-1246/).

In [3]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

books_url = "https://object.pouta.csc.fi/OPUS-Books/v1/moses/en-nl.txt.zip"
book_id = "Doyle_Arthur_Conan-Sign_of_Four.xml.gz"

with (
    urlopen(books_url, timeout=60) as download,
    ZipFile(BytesIO(download.read())) as books,
):
    english = books.read("Books.en-nl.en").decode("utf-8").split("\n")
    dutch = books.read("Books.en-nl.nl").decode("utf-8").split("\n")
    alignments = books.read("Books.en-nl.ids").decode("utf-8").split("\n")

pairs = []
for source, reference, alignment in zip(english, dutch, alignments, strict=True):
    if not alignment.startswith(f"en/{book_id}\t"):
        continue
    _, _, source_id, reference_id = alignment.split("\t")
    # The story starts at s6. Chapter II starts at s177.
    if not source_id[1:].isdigit() or not 6 <= int(source_id[1:]) < 177:
        continue
    if not reference_id[1:].isdigit() or not reference.strip() or not source.strip():
        continue
    if source.startswith("[") or reference.startswith("["):
        continue
    pairs.append((source.strip(), reference.strip()))

if not 1 <= sentence_count <= len(pairs):
    raise ValueError(f"Choose a sentence count between 1 and {len(pairs)}.")

sources, references = map(list, zip(*pairs[:sentence_count], strict=True))
print(f"Loaded {len(sources)} English–Dutch pairs.")

Loaded 100 English–Dutch pairs.


## 3. Translate each sentence twice

We use the short English/Dutch prompt from the training data. The model sees one source sentence at a time and always chooses its most likely next token. We switch the adapter off for the first run and on for the second. Both runs score only the text generated after the prompt.

In [4]:
import torch
from peft import PeftConfig, PeftModel
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU in Colab and run the notebook again.")

adapter = PeftConfig.from_pretrained(adapter_dir)
base_id = adapter.base_model_name_or_path

config = AutoConfig.from_pretrained(base_id, revision=adapter.revision)
if getattr(config, "quantization_config", None):
    # The T4 uses float16 for the arithmetic in the 4-bit model.
    config.quantization_config["bnb_4bit_compute_dtype"] = "float16"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, padding_side="left")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_id,
    revision=adapter.revision,
    config=config,
    dtype=torch.float16,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

generation = GenerationConfig(
    do_sample=False,
    max_new_tokens=192,
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
print(f"Loaded {base_id} and the saved adapter.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Loaded unsloth/Llama-3.2-3B-Instruct-bnb-4bit and the saved adapter.


In [5]:
from tqdm.auto import tqdm


@torch.inference_mode()
def translate(sources, label, batch_size=8):
    translations = []
    reached_limit = 0
    for offset in tqdm(range(0, len(sources), batch_size), desc=label):
        messages = [
            [{"role": "user", "content": f"English: {source}\nDutch"}]
            for source in sources[offset : offset + batch_size]
        ]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            padding=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
        output = model.generate(**inputs, generation_config=generation)
        new_tokens = output[:, inputs["input_ids"].shape[1] :]
        if new_tokens.shape[1] == generation.max_new_tokens:
            eos_ids = generation.eos_token_id
            eos_ids = [eos_ids] if isinstance(eos_ids, int) else eos_ids
            stop_ids = {generation.pad_token_id, *eos_ids}
            reached_limit += sum(
                token not in stop_ids for token in new_tokens[:, -1].tolist()
            )
        translations.extend(
            text.strip()
            for text in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        )

    print(
        f"{label}: {translations.count('')} empty outputs, {reached_limit} hit the length limit."
    )
    return translations


with model.disable_adapter():
    zero_shot = translate(sources, "Zero-shot")
fine_tuned = translate(sources, "Fine-tuned")

Zero-shot:   0%|          | 0/13 [00:00<?, ?it/s]

`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072, 'do_sample': True, 'temperature': 0.6, 'top_p': 0.9}. If this is not desired, please set these values explicitly.


Zero-shot: 0 empty outputs, 1 hit the length limit.


Fine-tuned:   0%|          | 0/13 [00:00<?, ?it/s]

Fine-tuned: 0 empty outputs, 0 hit the length limit.


## 4. Compare the scores and read the translations

**BLEU** measures overlap with the reference in words and short word sequences, with a penalty for overall output that is too short. **chrF** measures overlap in character sequences and gives some credit for partial word matches. SacreBLEU computes both scores across the full sample. Higher scores indicate closer matches to this reference.

A good Dutch translation can use different words from the reference. Read the examples alongside the scores and look for mistranslations, omissions, and changes in tone or register. These results describe Chapter I with this prompt. The sentences were held out from our fine-tuning exercise, and the book may have been seen during the original model’s training.

In [6]:
import pandas as pd
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF

systems = {"Zero-shot": zero_shot, "Fine-tuned": fine_tuned}
metrics = {"BLEU": BLEU(), "chrF": CHRF()}
scores = {
    name: {
        metric_name: metric.corpus_score(translations, [references]).score
        for metric_name, metric in metrics.items()
    }
    for name, translations in systems.items()
}
display(pd.DataFrame.from_dict(scores, orient="index").round(2))

for metric_name, metric in metrics.items():
    print(f"{metric_name} settings: {metric.get_signature()}")

,BLEU,chrF
Zero-shot,10.87,37.65
Fine-tuned,10.46,36.26


BLEU settings: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
chrF settings: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0


In [7]:
examples = pd.DataFrame(
    {
        "English source": sources,
        "Dutch reference": references,
        "Zero-shot": zero_shot,
        "Fine-tuned": fine_tuned,
    },
    index=range(1, len(sources) + 1),
)
with pd.option_context("display.max_colwidth", None):
    display(examples.head(5))

,English source,Dutch reference,Zero-shot,Fine-tuned
1,Sherlock Holmes took his bottle from the corner of the mantel-piece and his hypodermic syringe from its neat morocco case.,Sherlock Holmes nam zijn flesch van den schoorsteenmantel en zijn werktuig voor onderhuidsche inspuitingen uit zijn marokijnen foudraal.,Sherlock Holmes pakte zijn fles uit de hoek van het mantelpijp en zijn hypodermische naald uit haar nette morocco kast.,Sherlock Holmes pakte zijn fles uit de hoek van het mantelpaardje en zijn injectiespuit van de nette morocco doos.
2,"With his long, white, nervous fingers he adjusted the delicate needle, and rolled back his left shirt-cuff.","Met zijne lange, witte, zenuwachtige vingers bracht hij de fijne naald in orde, en schoof de linkermouw van zijn overhemd omhoog.","Met zijn lange, witte, nerveuze vingers verfijnde hij de zorgvuldige naald, en rolde hij zijn linker schortenpak omhoog.","Met zijn lange, witte, zenuwachtige vingers zorgde hij ervoor dat de scherpere naald precies in het juiste punt stak, en rolde zijn linkse mouw terug."
3,For some little time his eyes rested thoughtfully upon the sinewy forearm and wrist all dotted and scarred with innumerable puncture-marks.,"Gedurende eenigen tijd bleven zijne oogen nadenkend op den gespierden voorarm en pols gericht, die met ontelbare teekenen van inprikkingen als bezaaid waren.","Voor een klein ogenblik rustten zijn ogen nadenkend op het pezenbeen en de pols, die allemaal bevlekt waren met een onmetelijke hoeveelheid prikkers.",Even een paar minuten lang bleef hij stilzitten en naar het ruggenmerg en de elleboog en de hand met al die vele litte wondjes staren.
4,"Finally he thrust the sharp point home, pressed down the tiny piston, and sank back into the velvet-lined arm-chair with a long sigh of satisfaction.","Eindelijk trok hij de scherpe punt terug, drukte den kleinen piston omlaag en zonk met een langen zucht van voldoening in den fluweelen armstoel terug.","Ten slotsteen stak hij de scherpe punt in, drukte de kleine pistoon naar beneden en zonk terug in het velekozijn verlichte leunstoel met een lang zucht van tevredenheid.","Ten slotsteen stak hij de scherpste punt in het hout, drukte de kleine piston in en zonk terug in de armstoel met een diepe zucht van voldoening."
5,"Three times a day for many months I had witnessed this performance, but custom had not reconciled my mind to it.","Driemaal per dag was ik reeds gedurende verscheidene maanden getuige geweest van deze verrichting, maar de gewoonte was bij mij in dit opzicht geen tweede natuur geworden.","Drie keer per dag voor vele maanden had ik deze uitvoering getuigd, maar de gewoonte had mijn geest niet overgevallen.","Twice of elke dag voor maanden had ik dit alles getuigd, maar het gewendheid had mijn geest nog steeds niet overtuigd."


**Try it.** Pick a line where the two translations differ. Which conveys the meaning more accurately? Which voice feels more natural in Dutch? Would your judgment change if you read the surrounding lines?